# 5.6 Eksperimen Bertahap Gabungan 204 Fitur

Eksperimen **bertahap (ladder)**: matriks 204 fitur dipangkas perlahan 204 → 150 → 100 → 74 → 50 → 37. Setiap tahap diuji penuh (k=2..10), hasilnya dibandingkan, lalu disaring sampai optimal. Pemenang diteruskan ke notebook 5.7 via `best_config.json` + `labels_best.csv`.

Metode pemangkasan: **peringkat variansi** (fitur bervariansi tertinggi dipertahankan) — seragam di semua tahap agar sebanding. PCA tidak dipakai untuk ladder karena tahap terkecil (37) melebihi jumlah sampel (n=33): PCA mensyaratkan komponen ≤ n. Tidak ada hasil yang dikarang.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import json

SEED = 42
BASE = "../data/downloads/ekstraksi fitur"
OUT = "../data/processed"
IMG = "../data/images/eksperimen204"
LADDER = [204, 150, 100, 74, 50, 37]
os.makedirs(OUT, exist_ok=True)
os.makedirs(IMG, exist_ok=True)

## Fase 0 — Normalisasi kunci dan merge 204 fitur

Tiga file `ekstraksi_fitur_{co,no2,so2}.csv` digabungkan dengan **kunci nama daerah yang dinormalisasi** (strip, lower, rapatkan spasi) — bukan posisi baris. Observasi ganda daerah yang sama (mahasiswa berbeda) diagregasi mean dan didokumentasikan.

In [2]:
def norm_daerah(s):
    return " ".join(str(s).strip().lower().split())

frames, feats = {}, {}
for pol in ["co", "no2", "so2"]:
    df = pd.read_csv(f"{BASE}/ekstraksi_fitur_{pol}.csv")
    df["key"] = df["daerah"].map(norm_daerah)
    dup = df[df.duplicated("key", keep=False)].sort_values("key")
    if len(dup):
        print(f"{pol}: {dup['key'].nunique()} daerah diobservasi 2x — diagregasi mean:")
        print(dup[["id", "nama", "daerah"]].to_string(index=False))
    feat = [c for c in df.columns if c not in ("id", "nama", "daerah", "key")]
    assert len(feat) == 68, f"{pol}: {len(feat)} fitur, bukan 68!"
    g = df.groupby("key")
    agg = g[feat].mean()
    agg.insert(0, "daerah", g["daerah"].first())
    agg = agg.reset_index()
    frames[pol] = agg[["key", "daerah"] + feat].rename(
        columns={c: f"{c}_{pol}" for c in feat})
    feats[pol] = feat
    print(pol, df.shape, "fitur:", len(feat))

assert feats["co"] == feats["no2"] == feats["so2"], "urutan fitur beda antar polutan!"
m = frames["co"].merge(frames["no2"], on="key", how="outer", suffixes=("", "_drop"))
m = m.merge(frames["so2"], on="key", how="outer")
daerah_cols = [c for c in m.columns if c.startswith("daerah")]
m["daerah"] = m[daerah_cols].bfill(axis=1).iloc[:, 0]
m = m.drop(columns=[c for c in daerah_cols if c != "daerah"])
feat204 = [c for c in m.columns if c.endswith(("_co", "_no2", "_so2"))]
print("kolom fitur gabungan:", len(feat204))
print("baris tak lengkap:", m.loc[~m[feat204].notna().all(axis=1), "daerah"].tolist())
full = m[m[feat204].notna().all(axis=1)].copy().reset_index(drop=True)
full[["daerah"] + feat204].to_csv(f"{OUT}/clustering_204.csv", index=False)
with open(f"{OUT}/merge_report.txt", "w") as f:
    f.write(f"baris_total={len(m)}\nbaris_lengkap={len(full)}\nfitur=204\nseed={SEED}\n")
    f.write("agregasi=mean per daerah untuk observasi ganda\n")
    f.write("drop_karena_tak_lengkap=" + ";".join(
        m.loc[~m[feat204].notna().all(axis=1), "daerah"].tolist()) + "\n")
print("tersimpan:", f"{OUT}/clustering_204.csv", full.shape)

co: 2 daerah diobservasi 2x — diagregasi mean:
 id                       nama             daerah
 25            Laila Maghfiroh   Kamal, Bangkalan
 41 Muhammad Zaidan Nabil Rafi   Kamal, Bangkalan
 29            M. Fatihul Umam Kwanyar, Bangkalan
 34             Fahdimas akmal Kwanyar, Bangkalan
co (37, 72) fitur: 68
no2: 2 daerah diobservasi 2x — diagregasi mean:
 id                       nama             daerah
 12            Laila Maghfiroh   Kamal, Bangkalan
 25 Muhammad Zaidan Nabil Rafi   Kamal, Bangkalan
 57            M. Fatihul Umam Kwanyar, Bangkalan
 60             Fahdimas akmal Kwanyar, Bangkalan
no2 (37, 72) fitur: 68
so2: 2 daerah diobservasi 2x — diagregasi mean:
 id                       nama             daerah
 25            Laila Maghfiroh   Kamal, Bangkalan
 42 Muhammad Zaidan Nabil Rafi   Kamal, Bangkalan
 29            M. Fatihul Umam Kwanyar, Bangkalan
 34             Fahdimas akmal Kwanyar, Bangkalan
so2 (37, 72) fitur: 68
kolom fitur gabungan: 204
baris tak len

## Fase 1 — Studi interpolasi (linear vs cubic spline vs polinomial d=3)

Didefinisikan eksplisit agar tidak tumpang tindih. Kriteria: galat di titik valid + kehalusan + visual pada deret mentah Jabon.

In [3]:
from scipy.interpolate import CubicSpline

RAW = "../data/downloads/jabon_pollutants_data"
rows = []
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
for ax, pol in zip(axes, ["NO2", "CO", "SO2"]):
    s = pd.read_csv(f"{RAW}/jabon_{pol}.csv", parse_dates=["date"]).set_index("date")[pol]
    y = s.to_numpy(dtype=float)
    x = np.arange(len(y))
    valid = ~np.isnan(y)
    lin = pd.Series(y, index=x).interpolate(method="linear", limit_direction="both").to_numpy()
    spl = CubicSpline(x[valid], y[valid], bc_type="natural")(x)
    ply = np.polyval(np.polyfit(x[valid], y[valid], 3), x)
    smooth = lambda v: float(np.mean(np.abs(np.diff(v, 2))))
    rows.append({"polutan": pol, "n_missing": int((~valid).sum()),
                 "rmse_valid_linear": float(np.sqrt(np.mean((lin[valid] - y[valid]) ** 2))),
                 "rmse_valid_spline": float(np.sqrt(np.mean((spl[valid] - y[valid]) ** 2))),
                 "rmse_valid_poly3": float(np.sqrt(np.mean((ply[valid] - y[valid]) ** 2))),
                 "smooth_linear": smooth(lin), "smooth_spline": smooth(spl), "smooth_poly3": smooth(ply)})
    ax.plot(x, y, ".", ms=2, alpha=0.4, label="mentah")
    ax.plot(x, lin, lw=1, label="linear")
    ax.plot(x, spl, lw=1, label="cubic spline")
    ax.plot(x, ply, lw=1, label="polinomial d=3")
    ax.set_title(f"{pol} — missing {int((~valid).sum())} dari {len(y)}")
    ax.legend(fontsize=8, ncol=4)
fig.tight_layout()
fig.savefig(f"{IMG}/fig_interpolasi_perbandingan.png", dpi=100)
interp_df = pd.DataFrame(rows)
interp_df.to_csv(f"{OUT}/interpolasi_stats.csv", index=False)
print(interp_df.to_string(index=False))

polutan  n_missing  rmse_valid_linear  rmse_valid_spline  rmse_valid_poly3  smooth_linear  smooth_spline  smooth_poly3
    NO2         48                0.0       1.899970e-22          0.000011       0.000011       0.000012  7.846112e-10
     CO         44                0.0       0.000000e+00          0.003911       0.002857       0.003002  3.828050e-07
    SO2         40                0.0       0.000000e+00          0.000270       0.000273       0.000278  2.265023e-08


## Fase 2 — Pemangkasan bertahap 204 → 150 → 100 → 74 → 50 → 37

Setiap tahap mempertahankan fitur bervariansi tertinggi (variansi dihitung sebelum standardisasi). Kolom `retained_var` = proporsi total variansi yang dipertahankan tahap itu — inilah "hasil yang ditunjukkan" tiap pemangkasan.

In [4]:
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(f"{OUT}/clustering_204.csv")
feat204 = [c for c in df.columns if c != "daerah"]
Xraw = df[feat204].to_numpy()
var_rank = np.argsort(Xraw.var(axis=0))[::-1]
total_var = float(Xraw.var(axis=0).sum())
print(f"matriks: {Xraw.shape[0]} daerah x {Xraw.shape[1]} fitur")

lad_rows = []
for d in LADDER:
    keep = feat204 and sorted(var_rank[:d].tolist())
    cols = [feat204[i] for i in keep]
    Xs = StandardScaler().fit_transform(df[cols].to_numpy())
    retained = float(Xraw[:, var_rank[:d]].var(axis=0).sum() / total_var)
    pd.DataFrame(Xs).to_csv(f"{OUT}/X_lad_{d}.csv", index=False)
    lad_rows.append({"dimensi": d, "n_fitur": len(cols), "retained_var": round(retained, 4)})
    print(f"tahap {d}: {len(cols)} fitur, retained_var={retained:.4f}")
lad_df = pd.DataFrame(lad_rows)
lad_df.to_csv(f"{OUT}/ladder_table.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lad_df["dimensi"], lad_df["retained_var"], marker="o")
for _, r in lad_df.iterrows():
    ax.annotate(str(int(r['dimensi'])), (r['dimensi'], r['retained_var']), fontsize=8)
ax.set_xlabel("dimensi tahap"); ax.set_ylabel("proporsi variansi dipertahankan")
ax.set_title("Hasil tiap pemangkasan bertahap")
fig.tight_layout(); fig.savefig(f"{IMG}/fig_ladder_var.png", dpi=100)

matriks: 33 daerah x 204 fitur
tahap 204: 204 fitur, retained_var=1.0000
tahap 150: 150 fitur, retained_var=1.0000
tahap 100: 100 fitur, retained_var=1.0000
tahap 74: 74 fitur, retained_var=1.0000
tahap 50: 50 fitur, retained_var=1.0000
tahap 37: 37 fitur, retained_var=1.0000


## Fase 3 — Grid penuh + penyaringan sampai optimal

Setiap tahap × k=2..10 diuji (`random_state=42, n_init=10, max_iter=300`) — total 54 konfigurasi, semua disimpan. Penyaringan: (a) terbaik mentah per tahap; (b) terbaik multi-cluster (k≥3), karena seluruh konfigurasi mengandung 1 singleton outlier struktural sehingga saringan ukuran minimum kosong. Pemenang = silhouette tertinggi lapis (b); lapis mentah dilaporkan transparan sebagai pembanding.

In [5]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

res = []
for d in LADDER:
    Xd = pd.read_csv(f"{OUT}/X_lad_{d}.csv").to_numpy()
    for k in range(2, 11):
        km = KMeans(n_clusters=k, random_state=SEED, n_init=10, max_iter=300)
        lab = km.fit_predict(Xd)
        sil = float(silhouette_score(Xd, lab))
        sizes = pd.Series(lab).value_counts().sort_index()
        res.append({"dimensi": d, "k": k, "silhouette": sil,
                    "inertia": float(km.inertia_), "seed": SEED,
                    "min_size": int(sizes.min()),
                    "ukuran_cluster": ";".join(map(str, sizes.tolist()))})
grid = pd.DataFrame(res)
grid.to_csv(f"{OUT}/clustering_grid_results.csv", index=False)
print(f"total konfigurasi: {len(grid)}")
print(grid.sort_values("silhouette", ascending=False).head(10).to_string(index=False))

per_stage = grid.loc[grid.groupby("dimensi")["silhouette"].idxmax()]
print("\nTERBAIK MENTAH per tahap ( mayoritas k=2 + 1 singleton outlier ):")
print(per_stage[["dimensi", "k", "silhouette", "ukuran_cluster"]].to_string(index=False))
multi = grid[grid["k"] >= 3].sort_values("silhouette", ascending=False)
print("\nTERBAIK MULTI-CLUSTER (k>=3):")
print(multi.head(8)[["dimensi", "k", "silhouette", "ukuran_cluster"]].to_string(index=False))

win = multi.iloc[0]
best = {"dimensi": int(win["dimensi"]), "k": int(win["k"]),
        "silhouette": float(win["silhouette"]), "inertia": float(win["inertia"]),
        "seed": SEED, "aturan": "silhouette tertinggi dengan k>=3 (struktur multi-cluster)"}
with open(f"{OUT}/best_config.json", "w") as f:
    json.dump(best, f, indent=2)
Xd = pd.read_csv(f"{OUT}/X_lad_{best['dimensi']}.csv").to_numpy()
lab = KMeans(n_clusters=best["k"], random_state=SEED, n_init=10,
             max_iter=300).fit_predict(Xd)
pd.DataFrame({"daerah": df["daerah"].tolist(), "cluster": lab}).to_csv(
    f"{OUT}/labels_best.csv", index=False)
print("\nPEMENANG (diteruskan ke 5.7):", best)

piv = grid.pivot(index="dimensi", columns="k", values="silhouette").loc[LADDER]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im = axes[0].imshow(piv.to_numpy(), aspect="auto", cmap="YlGn")
axes[0].set_xticks(range(len(piv.columns))); axes[0].set_xticklabels(piv.columns)
axes[0].set_yticks(range(len(piv.index))); axes[0].set_yticklabels(piv.index)
axes[0].set_xlabel("k"); axes[0].set_ylabel("dimensi tahap")
axes[0].set_title("Heatmap silhouette (tahap × k)")
fig.colorbar(im, ax=axes[0], label="silhouette")
for d in LADDER:
    g = grid[grid["dimensi"] == d].sort_values("k")
    axes[1].plot(g["k"], g["inertia"], marker="o", ms=3, label=f"dim {d}")
axes[1].set_title("Elbow (inertia) per tahap"); axes[1].set_xlabel("k")
axes[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(f"{IMG}/fig_heatmap_elbow.png", dpi=100)

total konfigurasi: 54
 dimensi  k  silhouette     inertia  seed  min_size ukuran_cluster
      37  2    0.905455  224.035543    42         1           32;1
      50  2    0.878849  369.042144    42         1           32;1
      37  3    0.837483   61.596141    42         1         31;1;1
     100  2    0.831999  985.867029    42         1           32;1
      74  2    0.823672  818.174356    42         1           32;1
     150  2    0.788985 1953.332667    42         1           32;1
      50  3    0.780174  140.647087    42         1         31;1;1
     204  2    0.753039 3165.070715    42         1           32;1
      37  4    0.724793   31.695788    42         1       30;1;1;1
      74  3    0.657899  516.607912    42         1         31;1;1

TERBAIK MENTAH per tahap ( mayoritas k=2 + 1 singleton outlier ):
 dimensi  k  silhouette ukuran_cluster
      37  2    0.905455           32;1
      50  2    0.878849           32;1
      74  2    0.823672           32;1
     100  2    0.8

## Visualisasi — Peta kelengkapan data per daerah × polutan

Hijau = daerah punya 68 fitur polutan tersebut; merah = tidak ada (baris merah penuh = daerah gugur dari matriks 204).

In [6]:
def norm2(s):
    return " ".join(str(s).strip().lower().split())

keys, lab_of = {}, {}
for pol in ["co", "no2", "so2"]:
    d = pd.read_csv(f"{BASE}/ekstraksi_fitur_{pol}.csv")
    for _, r in d.iterrows():
        k = norm2(r["daerah"])
        keys.setdefault(k, {})[pol] = True
        lab_of.setdefault(k, r["daerah"])
cov = pd.DataFrame([{"daerah": lab_of[k], "CO": int("co" in v),
                     "NO2": int("no2" in v), "SO2": int("so2" in v)}
                    for k, v in sorted(keys.items())])
fig, ax = plt.subplots(figsize=(6, 12))
ax.imshow(cov[["CO", "NO2", "SO2"]].to_numpy(), aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["CO", "NO2", "SO2"])
ax.set_yticks(range(len(cov))); ax.set_yticklabels(cov["daerah"], fontsize=7)
ax.set_title("Kelengkapan 68 fitur per daerah (hijau=ada)")
fig.tight_layout(); fig.savefig(f"{IMG}/fig_coverage_daerah.png", dpi=100)
print(f"total label: {len(cov)}, lengkap 3 polutan: {int((cov[['CO','NO2','SO2']].sum(axis=1)==3).sum())}")

total label: 37, lengkap 3 polutan: 33


## Visualisasi — Diagram silhouette per sampel (pemenang)

Setiap batang = satu daerah. Batang panjang = sangat cocok di clusternya; pendek/negatif = ambigu.

In [7]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples

best = json.load(open(f"{OUT}/best_config.json"))
Xd = pd.read_csv(f"{OUT}/X_lad_{best['dimensi']}.csv").to_numpy()
lab_best = KMeans(n_clusters=best["k"], random_state=SEED, n_init=10,
                  max_iter=300).fit_predict(Xd)
s_vals = silhouette_samples(Xd, lab_best)
order = np.argsort(lab_best * 100 + s_vals)
names = df["daerah"].tolist()
cmap = plt.get_cmap("tab10")
bar_colors = [cmap(int(lab_best[i]) % 10) for i in order]
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(s_vals)), s_vals[order], color=bar_colors)
ax.axvline(s_vals.mean(), color="black", ls="--", lw=1, label="rata-rata")
ax.set_yticks(range(len(s_vals)))
ax.set_yticklabels([names[i] for i in order], fontsize=6)
ax.set_xlabel("s(i)")
ax.set_title(f"Silhouette per daerah (dim {best['dimensi']}, k={best['k']})")
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(f"{IMG}/fig_silhouette_samples.png", dpi=100)

## Visualisasi — Sebaran cluster di bidang PCA-2D + fitur pembeda

204 dimensi diringkas ke PC1 vs PC2 agar terlihat mata; diagram batang menunjukkan 10 fitur paling pembeda antar cluster pemenang.

In [8]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

dfm = pd.read_csv(f"{OUT}/clustering_204.csv")
Xm = StandardScaler().fit_transform(dfm[[c for c in dfm.columns if c != "daerah"]].to_numpy())
P2 = PCA(n_components=2, random_state=SEED).fit_transform(Xm)
kbest = len(set(lab_best))
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(P2[:, 0], P2[:, 1], c=lab_best, s=45, cmap="tab10", alpha=0.85)
fig.colorbar(sc, ax=ax, label="cluster")
for i in range(len(dfm)):
    if s_vals[i] < 0.3:
        ax.annotate(dfm.loc[i, "daerah"], (P2[i, 0], P2[i, 1]), fontsize=7)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title(f"Pemenang (dim {best['dimensi']}, k={best['k']}) di bidang PCA-2D")
ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(f"{IMG}/fig_cluster_pca2d.png", dpi=100)

means = [Xm[lab_best == k].mean(axis=0) for k in range(kbest)]
spread = pd.Series(np.max(means, axis=0) - np.min(means, axis=0),
                   index=[c for c in dfm.columns if c != "daerah"])
top = spread.sort_values(ascending=False).head(10)
fig2, ax2 = plt.subplots(figsize=(9, 4))
ax2.barh(top.index[::-1], top.values[::-1])
ax2.set_xlabel("rentang mean antar cluster (terstandardisasi)")
ax2.set_title("10 fitur paling pembeda")
fig2.tight_layout(); fig2.savefig(f"{IMG}/fig_top_pembeda.png", dpi=100)
print(top.to_string())

lpcc_co                  7.261136
calc_centroid_co         5.845159
distance_co              5.836457
median_frequency_co      5.835470
spectral_distance_so2    5.834463
spectral_distance_co     5.834120
calc_min_so2             5.834100
wavelet_abs_mean_co      5.833762
median_abs_diff_so2      5.833677
auc_co                   5.833664


## Kesimpulan 5.6 dan cara menjalankan ulang

- Pemenang tersimpan di `data/processed/best_config.json` + `labels_best.csv` dan **dibaca notebook 5.7** (peta memakai cluster dan fitur tahap pemenang).
- Tambah daerah ke `ekstraksi_fitur_*.csv` lalu jalankan ulang: ladder, grid, saringan, dan peta ikut mutakhir.